In [1]:
import numpy as np
import matplotlib.pyplot as plt
import stim
import pymatching
import sinter
from typing import List
import time
import pandas as pd
from IPython.display import HTML
import lmfit
import itertools
from lmfit import Model
from stimbposd import SinterDecoder_BPOSD, sinter_decoders
from stimbposd import BPOSD
from itertools import combinations
import galois
import ldpc
from ldpc import bposd_decoder

In [2]:
##Non Periodic boundary conditions

h=4   ###Height of grid
w=55  ###Width of grid, make odd k=27-6
H=[]
stabs=[]
packed=[]
for i in range(1,h):
    for j in range(0,w):          
        stab=[]

        stab.append([[i,j], [i-1,j-1],[i-1,j],[i-1,j+1]])

        sta=np.zeros([h,w])
        for pack in stab[0]:
            if pack[1] >= 0 and pack[1] < w:
                sta[pack[0]][pack[1]]=1
        sta=sta[::-1]
        final_sta=[sta[0],sta[1][1:-1],sta[2][2:-2],sta[3][3:-3]]
        final_sta = np.concatenate(final_sta).tolist()
        if np.sum(final_sta) > 0:
            H.append(final_sta)

        
H=np.array(H).astype(int)




In [3]:
def gf2_rref(A):
    A = A.copy()
    m, n = A.shape
    row = 0
    pivots = []

    for col in range(n):
        pivot = None
        for r in range(row, m):
            if A[r, col] == 1:
                pivot = r
                break
        if pivot is None:
            continue

        A[[row, pivot]] = A[[pivot, row]]
        pivots.append(col)

        for r in range(m):
            if r != row and A[r, col] == 1:
                A[r] ^= A[row]

        row += 1
        if row == m:
            break

    return A, pivots

def nullspace_gf2(H):
    R, pivots = gf2_rref(H)
    n = H.shape[1]
    free = [i for i in range(n) if i not in pivots]

    G = []
    for f in free:
        v = np.zeros(n, dtype=int)
        v[f] = 1
        for i, p in enumerate(pivots):
            v[p] = R[i, f]
        G.append(v)

    return np.array(G)

G_LDPC = nullspace_gf2(H)   # 12 × 23

# -----------------------------
# Convert G to systematic form
# -----------------------------
def systematic_form(G):
    G = G.copy()
    rows, cols = G.shape
    r = 0
    col_perm = list(range(cols))

    for c in range(cols):
        if r >= rows:
            break
        for rr in range(r, rows):
            if G[rr, c]:
                G[[r, rr]] = G[[rr, r]]
                col_perm[r], col_perm[c] = col_perm[c], col_perm[r]
                G[:, [r, c]] = G[:, [c, r]]
                for rrr in range(rows):
                    if rrr != r and G[rrr, r]:
                        G[rrr] ^= G[r]
                r += 1
                break

    return G, col_perm

#G_sys, perm = systematic_form(G)

print(len(G_LDPC))



49


In [15]:
import ldpc.codes
import ldpc.code_util
d, number_code_words_sampled, lowest_weight_codewords = ldpc.code_util.estimate_code_distance(np.array(H), timeout_seconds = 15)
print(f"Code distance estimate, d <= {d} (no. codewords sampled: {number_code_words_sampled})")

Code distance estimate, d <= 12 (no. codewords sampled: 15227899)


In [19]:
code_supports = []
for j in range(G_LDPC.shape[1]):      # loop over columns
    rows = np.where(G_LDPC[:, j] == 1)[0]
    code_supports.append([int(r % 7) for r in rows])

print(code_supports)

[[0], [0, 1], [1, 2], [0, 2, 3], [1, 3, 4], [0, 2, 4, 5], [0, 1, 3, 5, 6], [1, 2, 4, 6, 0], [2, 3, 5, 0, 1], [3, 4, 6, 1, 2], [4, 5, 0, 2, 3], [5, 6, 1, 3, 4], [6, 0, 2, 4, 5], [0, 1, 3, 5, 6], [1, 2, 4, 6, 0], [2, 3, 5, 0, 1], [3, 4, 6, 1, 2], [4, 5, 0, 2, 3], [5, 6, 1, 3, 4], [6, 0, 2, 4, 5], [0, 1, 3, 5, 6], [1, 2, 4, 6, 0], [2, 3, 5, 0, 1], [3, 4, 6, 1, 2], [4, 5, 0, 2, 3], [5, 6, 1, 3, 4], [6, 0, 2, 4, 5], [0, 1, 3, 5, 6], [1, 2, 4, 6, 0], [2, 3, 5, 0, 1], [3, 4, 6, 1, 2], [4, 5, 0, 2, 3], [5, 6, 1, 3, 4], [6, 0, 2, 4, 5], [0, 1, 3, 5, 6], [1, 2, 4, 6, 0], [2, 3, 5, 0, 1], [3, 4, 6, 1, 2], [4, 5, 0, 2, 3], [5, 6, 1, 3, 4], [6, 0, 2, 4, 5], [0, 1, 3, 5, 6], [1, 2, 4, 6, 0], [2, 3, 5, 0, 1], [3, 4, 6, 1, 2], [4, 5, 0, 2, 3], [5, 6, 1, 3, 4], [6, 0, 2, 4, 5], [0, 1, 3, 5, 6], [1, 2, 4, 6], [2, 3, 5], [3, 4, 6], [4, 5], [5, 6], [6], [0], [1], [0, 2], [1, 3], [0, 2, 4], [1, 3, 5], [2, 4, 6], [3, 5, 0], [4, 6, 1], [5, 0, 2], [6, 1, 3], [0, 2, 4], [1, 3, 5], [2, 4, 6], [3, 5, 0], [4, 6, 

In [20]:
def all_unique(lst):
    return len(lst) == len(set(lst))

for i in range(len(code_supports)):
    if not all_unique(code_supports[i]):
        print("Failure at qubit", i)

# Pointy stabilizer has a partition number of 7